# 🎯 [MNT04_M06] 장비 센서 상태 실시간 모니터링 및 결함 징후 경보 대시보드 구현 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MNT04/mnt04_m06_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 장비정비·수리부속 (Level 3) |
| **미션 ID** | MNT04_M06 |
| **미션 주제** | 장비 센서 상태 실시간 모니터링 및 결함 징후 경보 대시보드 구현 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
항공·기갑 장비의 엔진과 구동계는 고장이 나기 전에 진동, 온도, 압력에 미세한 변화가 먼저 나타나지만, 고정 임계값만으로는 고부하 운용일을 이상으로 오인하거나 초기 열화를 놓쳐 수리부속을 제때 확보하지 못한다. 본 프로젝트는 고장 실적과 수리부속 소모를 연결해 정비와 부품 관리의 연계 문제를 학습하는 것을 목표로, 교육용 가상 기동체 48대의 센서 시계열과 부품 소모를 코드로 만들어 분석한다. 센서 데이터셋 정의, 잡음 제거와 FFT 특징량 추출, 임계값 Baseline과 부품 소모 파레토 분석, 오토인코더 기반 미세 변화 감지와 파생 피처 모델, 잔존수명(RUL) 예측과 이상 신뢰도 점수, 설명가능한 경보 화면, 검사관 승인(HITL) 기반 조기 입고 시뮬레이터, 대용량 스트림 스트레스 시험, AAR 오경보 분석과 임계 재보정, 재학습을 포함한 통합 도구까지 단계별로 구현한다. 최종 목표는 고장 전에 이상 징후를 일찍 잡아 정비 시점과 수리부속 소요를 예고하되, 조기 입고 결정은 검사관이 근거를 보고 내리도록 지원하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📁 2. 실습 산출물(Deliverables) 디렉터리 구조 및 안내

> **💡 [학습 안내] 산출물 디렉터리 아키텍처**  
> 본 과제는 단순 일회성 노트북 실행에 그치지 않고, **실제 군수 정비 야전(정비대대, 군수지원사령부)에 즉시 도입 가능한 "완성형 국방 예지정비(PdM) 및 PBL 품목관리 도구"를 완성하는 실무 엔지니어링 과정**입니다. 각 미션의 **산출물(Deliverables)**에 명시된 디렉터리 경로는 실무 표준 소프트웨어 아키텍처에 해당합니다:

| 디렉토리 | 군 실무 소프트웨어 아키텍처 분류 | 주요 보관 산출물 | 관련 미션 |
| :--- | :--- | :--- | :---: |
| **`docs/`** | **국방 도메인 및 데이터 규격서** | 군수 도메인 분석서 및 데이터 딕셔너리(`docs/mnt04_logistics_domain_spec.md`), 표준 운영 절차 기술 매뉴얼(`docs/mnt04_operations_guide.md`) | 미션 1, 10 |
| **`src/etl/`** | **센서 데이터 정제 및 전처리 모듈** | 고주파 센서 신호 잡음 제거, 결측치 보정 및 FFT 특징 추출 파이프라인(`src/etl/mnt04_data_cleansing.py`) | 미션 2 |
| **`notebooks/`** | **탐색적 데이터 분석(EDA) & Baseline** | 3-시그마 임계값 Baseline 및 부품 소모 파레토 분석(`notebooks/mnt04_eda_analysis.ipynb`) | 미션 3 |
| **`src/features/`** | **전술 피처 엔지니어링** | 오토인코더 기반 미세 진동 변화 감지 및 파생 피처 엔지니어링 모듈(`src/features/mnt04_feature_engineering.py`) | 미션 4 |
| **`src/models/`** | **AI 예측 & 잔존수명(RUL) 엔진** | 부품 RUL 예측 앙상블 모델 및 종합 상태 이상 신뢰도 엔진(`src/models/mnt04_advanced_model.py`) | 미션 5 |
| **`src/explainability/`** | **XAI 설명가능성 모듈** | 오토인코더 재구성 오차 분해 및 XAI 원인 설명기(`src/explainability/mnt04_xai_explainer.py`) | 미션 6 |
| **`src/optimization/`** | **HITL 조기 입고 시뮬레이터** | 검사관 HITL 승인 큐, 감사 추적 및 조기 정비 시뮬레이터(`src/optimization/mnt04_simulator.py`) | 미션 7 |
| **`src/dashboard/`** | **스트림 주입 & 관제 대시보드** | 대용량 센서 스트림 주입 스트레스 시험 및 실시간 관제 대시보드(`src/dashboard/mnt04_app.py`) | 미션 8 |
| **`reports/`** | **검증 및 AAR 리포트** | 사후검토(AAR) 오경보 원인 분석 및 임계 파라미터 재보정 ROI 보고서(`reports/mnt04_roi_analysis.md`) | 미션 9 |
| **`pipeline/`** | **MLOps 자동 재학습** | 신규 데이터 기반 챔피언-도전자 자동 재학습 및 E2E 통합 파이프라인(`pipeline/mnt04_retrain.py`) | 미션 10 |

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 06. 장비 센서 상태 실시간 모니터링 및 결함 징후 경보 대시보드 구현
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

장비 센서 상태 실시간 모니터링 및 결함 경보 대시보드를 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **장비 센서 상태를 개체별로 모니터링해 결함 경보를 띄우는 정비병용 화면을 구성하고, 경보 건마다 기여도 상위 3개 인자와 자연어 요약을 자동 산출하는 설명가능성(XAI) 모듈을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **오토인코더 재구성 오차를 입력 인자별로 나누어 경보 건마다 상위 3개 인자를 뽑고, 1순위 인자를 정상값으로 되돌렸을 때 점수가 내려가는지로 근거 유효성을 점검 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/explainability/mnt04_xai_explainer.py 및 XAI 리포트` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군수 장교/군무원이 신뢰하고 조치할 수 있는 설명의 명확성

#### 미션 PASS 조건

- 정량 기준: 주요 예측 건당 기여도 상위 3개 인자 자동 산출률 100%


- 진행 기준: SHAP summary plot 및 waterfall plot 생성 파이프라인 완성


- 검증 방법: 실무자 대상 설명 근거 유용성 평가 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
# ============================================================================
# [미션 06 산출물 파일 연계]: src/explainability/mnt04_xai_explainer.py
# (선택) 로컬 모듈 파일로 추출 저장 시: %%writefile src/explainability/mnt04_xai_explainer.py
# ============================================================================
"""장비 센서 상태를 개체별로 모니터링해 이상 신호를 경보로 띄우는 정비병용 화면(상태표·점검 카드)을 구성하고, 오토인코더 재구성 오차에서 경보 건마다 기여도 상위 3개 인자를 자동 산출하며(근거 유효성 검증), 의심 부품과 자연어 요약 문장을 생성하는 설명가능성(XAI) 모듈을 구현한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kurtosis
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 기동체 48대의 엔진·구동계 진동·온도·압력 센서 자료 생성기 (실제 장비·부대·정비 실적 없음)
# 개체마다 정상 -> 열화 -> 고장(수명종료)까지 하루 1회 측정(진동 파형 256표본 + 온도 + 압력 + 부하)을 만든다.
# 열화 유형 4가지: bearing(베어링), gear(기어), oil_pump(오일 펌프), cooling(냉각 팬 모터)
# ---------------------------------------------------------------------------
FS, NS = 1024, 256                              # 가상 표본화 주파수(Hz), 파형 1개당 표본 수
TT = np.arange(NS) / FS
MODES = {  # 열화 유형: (발생 확률, 평균 수명(일), 교체 수리부속, 가상 단가지수)
    "bearing": (0.40, 160, "주 베어링 세트", 6.0), "gear": (0.25, 175, "구동 기어 세트", 9.0),
    "oil_pump": (0.20, 150, "오일 펌프", 4.0), "cooling": (0.15, 140, "냉각 팬 모터", 3.0)}
KIT_COST = 1.0                                  # 모든 정비에 함께 소모되는 씰·필터 키트 단가지수
BANDS = {"b50": (40, 60), "b100": (90, 110), "b150": (140, 156), "b165": (156, 176), "b200": (190, 215)}


def make_runs(n=48, seed=1):
    """반환: units(개체 정보), snap(일별 측정 요약), W(진동 파형 [측정 수, 256])"""
    rng = np.random.default_rng(seed)
    names = list(MODES)
    mode = rng.choice(names, n, p=[MODES[m][0] for m in names])
    life = np.clip(np.array([rng.normal(MODES[m][1], 18) for m in mode]), 90, 230).astype(int)
    start = np.sort(rng.integers(0, 240, n))
    units = pd.DataFrame({"unit": [f"U{i:02d}" for i in range(n)], "mode": mode, "life": life, "start": start, "base_load": rng.uniform(0.4, 0.75, n)})
    rows = []
    for u in units.itertuples():
        d = np.arange(u.life + 1)
        onset = 0.45 * u.life
        deg = np.clip((d - onset) / (u.life - onset), 0, 1) ** 1.5
        load = np.clip(u.base_load + rng.normal(0, 0.06, len(d)), 0.2, 1.0)
        hi = rng.random(len(d)) < 0.07                                            # 고부하 운용일(정상이지만 진동·온도 상승)
        load = np.where(hi, rng.uniform(0.95, 1.1, len(d)), load)
        rows.append(pd.DataFrame({"unit": u.unit, "day": d, "cal": u.start + d, "rul": u.life - d, "deg": deg, "load": load, "hi_load": hi, "mode": u.mode}))
    snap = pd.concat(rows, ignore_index=True)
    S = len(snap)
    snap["ambient"] = 15 + 12 * np.sin(2 * np.pi * snap["cal"] / 365) + rng.normal(0, 2, S)
    dg, ld, md = snap["deg"].values, snap["load"].values, snap["mode"].values
    isb, isg, iso, isc = (md == "bearing"), (md == "gear"), (md == "oil_pump"), (md == "cooling")
    snap["temp"] = 55 + 22 * ld + 0.5 * snap["ambient"] + 9 * dg * iso + 26 * dg * isc + 3 * dg * (isb | isg) + rng.normal(0, 1.5, S)
    snap["press"] = 450 - 12 * ld - 85 * dg * iso - 15 * dg * isc + rng.normal(0, 6, S)
    ph = rng.uniform(0, 2 * np.pi, (S, 3))
    a1, a2 = 1.0 * (0.5 + ld), 0.4 * (0.5 + ld)
    W = (a1[:, None] * np.sin(2 * np.pi * 50 * TT + ph[:, :1]) + a2[:, None] * np.sin(2 * np.pi * 100 * TT + ph[:, 1:2])
         + rng.normal(0, 0.35, (S, NS)))
    W += (0.9 * dg * isb)[:, None] * np.sin(2 * np.pi * 165 * TT + ph[:, 2:3])
    W += (0.8 * dg * isg)[:, None] * (np.sin(2 * np.pi * 150 * TT + ph[:, 2:3]) + 0.8 * np.sin(2 * np.pi * 100 * TT))
    W += (0.5 * dg * iso)[:, None] * np.sin(2 * np.pi * 200 * TT + ph[:, 2:3])
    k = rng.poisson(10 * dg * isb)                                                 # 베어링 열화: 충격성 펄스(첨도 상승)
    pos, amp = rng.integers(0, NS, (S, 30)), rng.normal(0, 3.0, (S, 30)) * dg[:, None]
    amp = amp * (np.arange(30)[None, :] < k[:, None])
    np.add.at(W, (np.repeat(np.arange(S), 30), pos.ravel()), amp.ravel())
    return units, snap, W


def extract(W):
    """진동 파형 -> 특징량: RMS, 첨도, 파고율, 최대-최소, FFT 대역 에너지(50/100/150/165/200Hz 부근)"""
    # [TODO : 학생 실습 6] 장비 센서 상태 실시간 모니터링 및 결함 징후 경보 대시보드 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
BASE = ["rms", "kurt", "crest", "p2p", "b50", "b100", "b150", "b165", "b200", "temp", "press"]
ROLL = ["rms", "kurt", "b150", "b165", "b200", "temp", "press"]


def build_features(snap, W):
    # [TODO : 학생 실습 6] 장비 센서 상태 실시간 모니터링 및 결함 징후 경보 대시보드 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
DERIVED = ["temp_res", "press_res", "rms_n", "hf_ratio"] + [c + "_m3" for c in ROLL] + ["rms_n_d", "temp_res_d", "press_res_d"]
FEATS = BASE + DERIVED
AE_COLS = ["rms_n", "kurt", "b150", "b165", "b200", "hf_ratio", "temp_res", "press_res"]


def split_units(units, n_train=24):
    return list(units["unit"][:n_train]), list(units["unit"][n_train:])


class AE:
    """오토인코더(입력 8 -> 10 -> 4 -> 10 -> 8): 정상 자료만 재구성하도록 학습하고, 재구성 오차를 이상 점수로 쓴다."""

    def __init__(self, cols=AE_COLS, q=0.99, seed=0):
        self.cols, self.q, self.seed = cols, q, seed

    def fit(self, healthy):
        self.sc = StandardScaler().fit(healthy[self.cols])
        Z = self.sc.transform(healthy[self.cols])
        self.net = MLPRegressor(hidden_layer_sizes=(10, 4, 10), max_iter=800, random_state=self.seed).fit(Z, Z)
        self.thr = float(np.quantile(self.err(healthy), self.q))
        return self

    def contrib(self, d):
        Z = self.sc.transform(d[self.cols])
        return (Z - self.net.predict(Z)) ** 2                 # 입력 항목별 재구성 오차(이상 기여도)

    def err(self, d):
        return self.contrib(d).mean(axis=1)


def latch(d, e, thr, k=2):
    """같은 개체에서 이상 점수가 k일 연속 기준을 넘으면 경보를 켜고 유지(래치)한다."""
    out, e = np.zeros(len(d), bool), np.asarray(e)
    for idx in d.reset_index(drop=True).groupby("unit").indices.values():
        run, on = 0, False
        for i in idx:
            run = run + 1 if e[i] > thr else 0
            on = on or run >= k
            out[i] = on
    return out

units, snap, W = make_runs()
D = build_features(snap, W)
tr_u, va_u = split_units(units)
tr, va = D[D["unit"].isin(tr_u)].reset_index(drop=True), D[D["unit"].isin(va_u)].reset_index(drop=True)
ae = AE().fit(tr[tr["deg"] == 0])
C = ae.contrib(va)                                         # 건 x 입력항목 기여도
e = C.mean(axis=1)
va["alarm"] = latch(va, e, ae.thr, k=2)
va["score"] = e / ae.thr

LABEL = {"rms_n": "부하 보정 진동 크기(RMS)", "kurt": "진동 충격성(첨도)", "b150": "150Hz 부근 진동(기어 맞물림 대역)", "b165": "165Hz 부근 진동(베어링 결함 대역)",
         "b200": "200Hz 부근 진동(펌프 대역)", "hf_ratio": "고주파/기본주파 에너지 비", "temp_res": "부하 대비 온도 상승", "press_res": "부하 대비 압력 저하"}
PART = {"b165": "주 베어링 세트", "kurt": "주 베어링 세트", "b150": "구동 기어 세트", "b200": "오일 펌프", "press_res": "오일 펌프", "temp_res": "냉각 팬 모터"}
TRUE_PART = {m: MODES[m][2] for m in MODES}
cols = ae.cols

# 1) 경보 건마다 기여도 상위 3개 인자 자동 산출
idx = np.where(va["alarm"].values)[0]
top3 = np.argsort(-C[idx], axis=1)[:, :3]
ok = sum(len(set(t)) == 3 for t in top3)
print(f"[교육용가상자료] 검증 {len(va_u)}대 {len(va)}건 중 경보 {len(idx)}건: 기여도 상위 3개 인자 산출 {ok}건 (산출률 {ok / len(idx):.0%}, 목표 100%: {'달성' if ok == len(idx) else '미달'})")

# 2) 근거 유효성: 1순위 인자를 정상 평균(표준화 0)으로 되돌리면 이상 점수가 낮아지는가
Z = ae.sc.transform(va[cols].iloc[idx])
Z2 = Z.copy()
Z2[np.arange(len(idx)), top3[:, 0]] = 0.0
e1, e2 = ((Z - ae.net.predict(Z)) ** 2).mean(axis=1), ((Z2 - ae.net.predict(Z2)) ** 2).mean(axis=1)
print(f"근거 유효성: 1순위 인자를 정상값으로 되돌리면 이상 점수가 낮아지는 비율 {(e2 < e1).mean():.1%} (평균 {e1.mean():.2f} -> {e2.mean():.2f})")

# 3) 의심 부품: 1순위 인자 -> 부품 매핑 (열화가 뚜렷한 경보에서 정답 비교)
sub = va.iloc[idx].copy()
sub["top1"] = [cols[t[0]] for t in top3]
sub["hint"] = sub["top1"].map(PART).fillna("미분류(점검 필요)")
sub["true_part"] = sub["mode"].map(TRUE_PART)
strong = sub[sub["deg"] >= 0.3]
print(f"의심 부품 추정 정확도(열화 0.3 이상 경보 {len(strong)}건): {(strong['hint'] == strong['true_part']).mean():.1%}")
print("  1순위 인자 분포: " + ", ".join(f"{LABEL[k].split('(')[0]} {v}" for k, v in sub["top1"].value_counts().head(4).items()))


def summarize(r, t):
    f = [LABEL[cols[i]] for i in t]
    part = PART.get(cols[t[0]], "원인 불명")
    return f"{part} 이상 가능성(이상 점수 {r['score']:.1f}배). 주요 근거: {f[0]}, {f[1]}, {f[2]}."


# 4) 정비병용 화면: 점검 카드와 상태표
print("\n[점검 카드] 개체 | 측정일 | 실제 열화 유형")
for m in MODES:
    k = sub[(sub["mode"] == m) & (sub["deg"] > 0.5)].index[0]
    j = list(sub.index).index(k)
    r = sub.loc[k]
    print(f" {r['unit']}  {int(r['day'])}일째  ({m})")
    print("   " + summarize(r, top3[j]))
    print("   근거 기여도: " + " / ".join(f"{LABEL[cols[i]].split('(')[0]}({C[idx[j], i]:.1f})" for i in top3[j]))

CAL = int(va.groupby('cal')['alarm'].sum().idxmax())               # 경보가 가장 많은 날을 가상 기준일로 쓴다
act = va[(va["cal"] == CAL)]
reg = {"정상": 0, "주의": 0, "경보": 0}
rows = []
for r in act.itertuples():
    st = "경보" if r.alarm else ("주의" if r.score > 0.7 else "정상")
    reg[st] += 1
for st in ("경보", "주의"):
    for r in act[(act["alarm"] if st == "경보" else (~act["alarm"] & (act["score"] > 0.7)))].itertuples():
        rows.append((r.unit, st, f"{r.score:.1f}", int(r.rul)))
print(f"\n[운용 상태표] 가상 기준일 {CAL}: 가동 중 {len(act)}대 -> " + ", ".join(f"{k} {v}대" for k, v in reg.items()))
for r in rows[:5]:
    print(f"  {r[0]}  {r[1]}  이상 점수 {r[2]}배  (실제 잔존 {r[3]}일)")

plt.figure(figsize=(8, 4))
for u in sub["unit"].unique()[:3]:
    g = va[va["unit"] == u]
    plt.plot(g["day"], g["score"], label=u)
plt.axhline(1.0, ls="--", color="k"); plt.yscale("log"); plt.xlabel("day"); plt.ylabel("anomaly score (x threshold)"); plt.legend(); plt.title("Monitoring view: anomaly score")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 06 예상 실행 결과 (Output)</b></summary>

```text
[교육용가상자료] 검증 24대 3886건 중 경보 1327건: 기여도 상위 3개 인자 산출 1327건 (산출률 100%, 목표 100%: 달성)
근거 유효성: 1순위 인자를 정상값으로 되돌리면 이상 점수가 낮아지는 비율 98.9% (평균 21.19 -> 6.57)
의심 부품 추정 정확도(열화 0.3 이상 경보 1167건): 99.3%
  1순위 인자 분포: 150Hz 부근 진동 603, 165Hz 부근 진동 338, 부하 대비 온도 상승 146, 부하 대비 압력 저하 129

[점검 카드] 개체 | 측정일 | 실제 열화 유형
 U27  131일째  (bearing)
   주 베어링 세트 이상 가능성(이상 점수 7.1배). 주요 근거: 165Hz 부근 진동(베어링 결함 대역), 150Hz 부근 진동(기어 맞물림 대역), 부하 대비 압력 저하.
   근거 기여도: 165Hz 부근 진동(36.6) / 150Hz 부근 진동(22.9) / 부하 대비 압력 저하(5.4)
 U26  150일째  (gear)
   구동 기어 세트 이상 가능성(이상 점수 10.4배). 주요 근거: 150Hz 부근 진동(기어 맞물림 대역), 165Hz 부근 진동(베어링 결함 대역), 진동 충격성(첨도).
   근거 기여도: 150Hz 부근 진동(60.3) / 165Hz 부근 진동(19.9) / 진동 충격성(10.9)
 U25  108일째  (oil_pump)
   오일 펌프 이상 가능성(이상 점수 6.0배). 주요 근거: 부하 대비 압력 저하, 200Hz 부근 진동(펌프 대역), 165Hz 부근 진동(베어링 결함 대역).
   근거 기여도: 부하 대비 압력 저하(29.3) / 200Hz 부근 진동(22.0) / 165Hz 부근 진동(4.7)
 U24  107일째  (cooling)
   냉각 팬 모터 이상 가능성(이상 점수 4.4배). 주요 근거: 부하 대비 온도 상승, 200Hz 부근 진동(펌프 대역), 진동 충격성(첨도).
   근거 기여도: 부하 대비 온도 상승(34.5) / 200Hz 부근 진동(3.9) / 진동 충격성(2.4)

[운용 상태표] 가상 기준일 308: 가동 중 18대 -> 정상 6대, 주의 1대, 경보 11대
  U26  경보  이상 점수 30.5배  (실제 잔존 1일)
  U28  경보  이상 점수 55.0배  (실제 잔존 5일)
  U31  경보  이상 점수 85.1배  (실제 잔존 1일)
  U32  경보  이상 점수 12.9배  (실제 잔존 14일)
  U33  경보  이상 점수 27.4배  (실제 잔존 14일)
```
</details>
